## Week 1 - Lab 2: The LLM Showdown

Today we compare multiple LLMs on the same question and use Gemini to judge which gives the best answer.

This demonstrates a core **agentic pattern**: using one LLM to evaluate the output of others.

### How this works

1. Gemini generates a challenging question
2. Multiple models answer it (Gemini Flash, Gemini Pro, optionally OpenAI/Claude/Ollama)
3. A judge LLM ranks the answers

The `openai` package doubles as a universal client — many providers expose an **OpenAI-compatible API**, so you just swap the `base_url` to talk to Gemini, DeepSeek, Groq, or Ollama.

In [ ]:
import os
import json
from dotenv import load_dotenv
from google import genai
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)

In [ ]:
# Check which keys are available

gemini_api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

print(f"Gemini key: {'✓ ' + gemini_api_key[:8] if gemini_api_key else '✗ not set'}")
print(f"OpenAI key: {'✓ ' + openai_api_key[:8] if openai_api_key else '✗ not set (optional)'}")
print(f"Anthropic key: {'✓ ' + anthropic_api_key[:7] if anthropic_api_key else '✗ not set (optional)'}")

In [ ]:
# Use Gemini's native SDK to generate the challenge question

client = genai.Client()

request = (
    "Come up with a challenging, nuanced question that I can ask multiple LLMs to evaluate their reasoning. "
    "Respond only with the question — no explanation."
)

question_response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=request
)
question = question_response.text
print(question)

In [ ]:
# We'll track competitors and their answers
competitors = []
answers = []

## Gemini models (via native SDK)

Let's get answers from two Gemini models first.

In [ ]:
# Gemini 2.0 Flash — fast and free

model_name = "gemini-2.0-flash"
response = client.models.generate_content(model=model_name, contents=question)
answer = response.text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Gemini 2.5 Flash — more capable, still in the free tier

model_name = "gemini-2.5-flash-preview-04-17"
response = client.models.generate_content(model=model_name, contents=question)
answer = response.text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

## Other providers (via OpenAI-compatible API)

The `openai` package supports swapping the `base_url` to point to any OpenAI-compatible endpoint.
This is a neat trick — one SDK, many providers.

In [ ]:
# OpenAI (if you have a key)

if openai_api_key:
    oai = OpenAI()
    model_name = "gpt-4o-mini"
    response = oai.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": question}]
    )
    answer = response.choices[0].message.content
    display(Markdown(answer))
    competitors.append(model_name)
    answers.append(answer)
else:
    print("Skipping OpenAI — key not set")

In [ ]:
# Gemini via OpenAI-compatible endpoint — shows that the compat layer also works
# Useful when you want one code path for multiple providers

gemini_compat = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
model_name = "gemini-2.0-flash (compat)"

response = gemini_compat.chat.completions.create(
    model="gemini-2.0-flash",
    messages=[{"role": "user", "content": question}]
)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

## Ollama (local models — optional)

Ollama runs models locally with an OpenAI-compatible API.

Install from [https://ollama.com](https://ollama.com), then pull a model:
```
ollama pull llama3.2
```
After installing, visit [http://localhost:11434](http://localhost:11434) — you should see "Ollama is running".

In [ ]:
# Uncomment to use Ollama (requires Ollama running locally)

# !ollama pull llama3.2

# ollama_client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# model_name = "llama3.2"

# response = ollama_client.chat.completions.create(
#     model=model_name,
#     messages=[{"role": "user", "content": question}]
# )
# answer = response.choices[0].message.content

# display(Markdown(answer))
# competitors.append(model_name)
# answers.append(answer)

print(f"Total competitors: {len(competitors)}")
print(competitors)

## Judgement time!

We'll use Gemini as the judge. It will rank all answers from best to worst.

In [ ]:
# Build the combined responses string for the judge

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index + 1}\n\n"
    together += answer + "\n\n"

print(together[:500], "...")

In [ ]:
judge_prompt = f"""You are judging a competition between {len(competitors)} AI models.
Each model was asked this question:

{question}

Your job: evaluate each response for clarity, accuracy, and depth of reasoning.
Rank them from best to worst.

Respond ONLY with JSON in this exact format (no markdown, no code blocks):
{{"results": ["best competitor number", "second best", ...]}}

Responses:

{together}
"""

In [ ]:
judge_response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=judge_prompt
)
print(judge_response.text)

In [ ]:
# Parse the JSON and display the final ranking

results_text = judge_response.text.strip()
# Strip any accidental markdown fences
if results_text.startswith("```"):
    results_text = results_text.split("\n", 1)[1].rsplit("```", 1)[0]

results_dict = json.loads(results_text)
ranks = results_dict["results"]

print("\n=== Final Rankings ===")
for position, result in enumerate(ranks):
    model = competitors[int(result) - 1]
    print(f"Rank {position + 1}: {model}")

## What patterns did we use?

This notebook demonstrated two foundational agentic patterns:

- **Parallelization** — sending the same task to multiple models simultaneously
- **LLM-as-Judge** — using a model to evaluate and rank other models' outputs

These patterns are everywhere in production AI systems: A/B testing prompts, quality gates, ensembling.

## Exercise

Which agentic pattern(s) did this lab use? Try adding another pattern:

- **Reflection**: after ranking, ask the losing model to critique the winner's answer and improve it
- **Self-consistency**: ask the same model the question 3 times and pick the most common answer
- **Chain of thought**: add `"Think step by step."` to the prompt and see if rankings change

Try at least one and see what changes!

In [ ]:
# Your experiment here!
